<a href="https://colab.research.google.com/github/SamashaHettiarachchi/AI_Challanges/blob/main/Fish_Weight_Linear_Regression_Annotated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fish Weight Prediction: Linear Regression from Scratch vs scikit-learn

In this notebook we predict the **weight of a fish** from its body measurements. The workflow has four parts:

1. Explore and clean the data.
2. Build **simple linear regression** (one feature) with **gradient descent written from scratch**.
3. Build **multiple linear regression** (five features) with the same code.
4. Check our results against **scikit-learn**.

## Dataset: Fish Market

159 fish from 7 species sold at a market.

| Column | Meaning |
|---|---|
| `Species` | Fish species (categorical) |
| `Weight` | Weight in grams (**target**: what we predict) |
| `Length1` | Vertical length (cm) |
| `Length2` | Diagonal length (cm) |
| `Length3` | Cross length (cm) |
| `Height` | Height (cm) |
| `Width` | Diagonal width (cm) |

For the simple model we use only `Length1`. For the multiple model we use all five measurements.

## 1. Load the Data

We read a public copy of the dataset directly from a URL with `pd.read_csv` and save it locally as `Fish.csv`. After this step the rest of the notebook only depends on the local file.

`df.shape` returns `(rows, columns)`. We expect `(159, 7)`: 159 fish and 7 columns.

In [49]:
# pandas is used to load and handle tabular data
import pandas as pd

# Public copy of the Fish Market dataset
url = "https://raw.githubusercontent.com/harika-bonthu/SupportVectorClassifier/main/datasets_229906_491820_Fish.csv"

# Read the CSV straight from the URL into a DataFrame
df = pd.read_csv(url)

# Save a local copy so the rest of the notebook can read "Fish.csv"
df.to_csv("Fish.csv", index=False)

# (rows, columns) -> expected (159, 7)
print(df.shape)
df.head()

(159, 7)


,Species,Weight,Length1,Length2,Length3,Height,Width
0,Bream,242.0,23.2,25.4,30.0,11.5200,4.0200
1,Bream,290.0,24.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,23.9,26.5,31.1,12.3778,4.6961
3,Bream,363.0,26.3,29.0,33.5,12.7300,4.4555
4,Bream,430.0,26.5,29.0,34.0,12.4440,5.1340


## 2. Import Libraries

| Library | What we use it for |
|---|---|
| `numpy` | Arrays and the math inside gradient descent |
| `pandas` | Holding the data as a table |
| `plotly.express` | Interactive plots |
| `pickle` | Saving the trained model to a file |
| `train_test_split` (scikit-learn) | Splitting the data into training and test sets |

In [50]:
import math
import numpy as np                 # numerical arrays and linear algebra
import pandas as pd                # tables (DataFrames)
import plotly.express as px        # interactive plots
import pickle                      # save / load the trained model

# Splits the data into training and test sets
from sklearn.model_selection import train_test_split

In [51]:
DATA_PATH = 'Fish.csv'   # change this if the file is stored somewhere else

# Load the saved CSV into a DataFrame called "dataset"
dataset = pd.read_csv(DATA_PATH)

print(dataset.shape)   # (159, 7): 159 fish, 7 columns
dataset.head()         # first 5 rows

(159, 7)


,Species,Weight,Length1,Length2,Length3,Height,Width
0,Bream,242.0,23.2,25.4,30.0,11.5200,4.0200
1,Bream,290.0,24.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,23.9,26.5,31.1,12.3778,4.6961
3,Bream,363.0,26.3,29.0,33.5,12.7300,4.4555
4,Bream,430.0,26.5,29.0,34.0,12.4440,5.1340


## 3. Data Exploration

Before building any model we check what the data looks like.

- `describe()` gives the count, mean, spread, minimum and maximum of each column.
- `isna().sum()` counts missing values in each column.
- `value_counts()` shows how many fish belong to each species.

**What we find:**
- There are **no missing values**.
- There are 7 species: Perch (56), Bream (35), Roach (20), Pike (17), Smelt (14), Parkki (11) and Whitefish (6).
- The minimum `Weight` is **0**. A fish with normal length, height and width cannot weigh 0 g, so this looks like a data-entry error. We deal with it in the next step.
- `Weight` ranges from 0 to 1650 g, while `Length1` ranges from about 7.5 to 59 cm. The two columns are on very different scales, which is one reason we standardize later.

In [52]:
# Summary statistics (count, mean, std, min, quartiles, max) of every numeric column
dataset.describe()

,Weight,Length1,Length2,Length3,Height,Width
count,159.000000,159.000000,159.000000,159.000000,159.000000,159.000000
mean,398.326415,26.247170,28.415723,31.227044,8.970994,4.417486
std,357.978317,9.996441,10.716328,11.610246,4.286208,1.685804
min,0.000000,7.500000,8.400000,8.800000,1.728400,1.047600
25%,120.000000,19.050000,21.000000,23.150000,5.944800,3.385650
50%,273.000000,25.200000,27.300000,29.400000,7.786000,4.248500
75%,650.000000,32.700000,35.500000,39.650000,12.365900,5.584500
max,1650.000000,59.000000,63.400000,68.000000,18.957000,8.142000


In [53]:
print(dataset.shape)                       # size of the table
print(dataset.isna().sum())                # number of missing values in each column
print(dataset['Species'].value_counts())   # how many fish of each species

(159, 7)
Species    0
Weight     0
Length1    0
Length2    0
Length3    0
Height     0
Width      0
dtype: int64
Species
Perch        56
Bream        35
Roach        20
Pike         17
Smelt        14
Parkki       11
Whitefish     6
Name: count, dtype: int64


## 4. Data Cleaning

We keep only the columns we need for the simple model (`Length1` and `Weight`), drop rows with missing values, and **remove the fish with `Weight = 0`**.

The shape changes from `(159, 7)` to `(158, 2)`: one fish removed and only two columns kept.

`data.describe()` on the cleaned data should now show a minimum weight of about 5.9 g instead of 0. Note that the earlier `dataset.describe()` still shows 0 because `dataset` is the original, uncleaned table; we cleaned a separate copy called `data`.

In [54]:
# Keep only the feature we use (Length1) and the target (Weight)
data = dataset[['Length1', 'Weight']]

# Drop rows that contain missing values (there are none, but this is a safe habit)
data = data.dropna()

# Remove the fish with Weight = 0: an impossible value, i.e. a data-entry error
data = data[data['Weight'] > 0]

print(f"Shape: {data.shape}")   # expected (158, 2)

Shape: (158, 2)


In [55]:
# Summary of the CLEANED data: Weight min should no longer be 0
data.describe()

,Length1,Weight
count,158.000000,158.000000
mean,26.293038,400.847468
std,10.011427,357.697796
min,7.500000,5.900000
25%,19.150000,121.250000
50%,25.300000,281.500000
75%,32.700000,650.000000
max,59.000000,1650.000000


## 5. Visualize the Relationship

**What the plot shows:**
- The relationship is **positive**: longer fish are heavier.
- It is **curved, not a straight line**. Weight grows faster than length. This makes sense because a fish is a 3-D object, so doubling its length multiplies its volume (and weight) by roughly eight.
- Species form different groups. For example, Pike (the long, thin fish) weigh less than other species of the same length.

A straight line cannot capture this curve perfectly, so we should expect some error from a linear model.

In [56]:
# Scatter plot of length vs weight, coloured by species (Weight = 0 row excluded)
px.scatter(dataset[dataset['Weight'] > 0], x='Length1', y='Weight', color='Species',
           template='seaborn', title='Length1 vs Weight')

## 6. Train/Test Split

We split the data into two parts:

- **Training set (80%, 126 fish):** the model learns from these.
- **Test set (20%, 32 fish):** kept hidden during training, used only to evaluate the model.

Think of it as an exam: if the model were graded on the same fish it studied, we could not tell whether it learned or just memorized. Testing on unseen fish tells us how well it generalizes.

`random_state=42` fixes the random shuffle, so the split is identical every time.

In [57]:
# Feature (Length1) and target (Weight) as NumPy arrays
X = data['Length1'].values
y = data['Weight'].values

# 80% of the fish for training, 20% for testing
# random_state=42 makes the split identical every time we run the notebook
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")

X_train: (126,), y_train: (126,)
X_test:  (32,), y_test:  (32,)


## 7. Standardize the Data

Standardization (z-score scaling) rewrites every value as "how many standard deviations away from the mean":

$$
x_{\text{scaled}} = \frac{x - \mu}{\sigma}
$$

**Why we do it:**
- Gradient descent converges faster and more stably when features are on similar, small scales.
- With several features, standardization puts them on the same scale so their weights can be compared.

**Important:** the mean $\mu$ and standard deviation $\sigma$ are computed on the **training set only** and then applied to the test set. Using test data to compute them would leak information from the test set into training (data leakage).

We also return and save the mean and std. We need them later to convert the learned weight back to centimetres and grams.

**Check:** after scaling, the training data has mean $\approx 0$ and std $= 1$. The saved original values are mean $\approx 26.08$ cm and std $\approx 9.91$ cm. (`-0.0000` is just a tiny floating-point rounding error.)

In [58]:
def standardize_data(X_train, X_test):
    """Z-score scaling: (x - mean) / std, using TRAINING statistics only."""

    # Mean and standard deviation are computed on the training set only,
    # so no information from the test set leaks into training
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)

    # Apply the same transformation to both sets
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    # Return mean and std too: we need them later to convert weights back to original units
    return X_train, X_test, mean, std


X_train, X_test, X_mean, X_std = standardize_data(X_train, X_test)

In [59]:
# After standardization the training data should have mean ~0 and std = 1
print(f"Train mean: {X_train.mean():.4f}")
print(f"Train std:  {X_train.std():.4f}")

# The original (cm) mean and std that we saved
print(f"Saved mean: {X_mean:.4f}, Saved std: {X_std:.4f}")

Train mean: -0.0000
Train std:  1.0000
Saved mean: 26.0825, Saved std: 9.9137


### Reshape the Data

The model expects a 2-D array of shape `(number of fish, number of features)`. Our `X` has shape `(126,)`, so we add a second dimension to get `(126, 1)`. With five features the shape will simply be `(126, 5)`, and the same code works for both.

In [60]:
# Turn shape (126,) into (126, 1): rows = fish, columns = features
# The model expects a 2-D array so the same code also works with many features
X_train = np.expand_dims(X_train, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

X_train: (126, 1)
X_test:  (32, 1)


## 8. Model Implementation: Linear Regression with Gradient Descent

**The model** is a straight line:

$$\hat{y} = w x + b$$

For our data: $\text{Weight} = w \cdot \text{Length1} + b$. Training means finding the best $w$ (slope) and $b$ (intercept).

**Cost function** (how wrong the model is, the mean squared error divided by 2):

$$J(w,b) = \frac{1}{2m} \sum_{i=1}^{m}\left(\hat{y}^{(i)} - y^{(i)}\right)^2$$

**Gradients** (how the cost changes when $w$ or $b$ change):

$$\frac{\partial J}{\partial w} = \frac{1}{m}\sum_{i=1}^{m}\left(\hat{y}^{(i)} - y^{(i)}\right)x^{(i)} \qquad \frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=1}^{m}\left(\hat{y}^{(i)} - y^{(i)}\right)$$

**Update rule** (take a small step downhill, $\alpha$ = learning rate):

$$w \leftarrow w - \alpha \frac{\partial J}{\partial w} \qquad b \leftarrow b - \alpha \frac{\partial J}{\partial b}$$

**One iteration** = forward pass (predict) -> compute cost -> compute gradients -> update $w$ and $b$. Repeating this many times moves the model toward the lowest cost. A good analogy is walking down a foggy hill: you cannot see the bottom, but you can feel which way the ground slopes and take a small step that way.

**Learning rate $\alpha$:** too large and the steps overshoot (the cost can grow); too small and training is very slow.

The class below also supports an early stop (`convergence_tol`) and saving/loading the learned parameters.

In [61]:
# Linear regression trained with batch gradient descent (works for 1 or many features)
class LinearRegression:

    # Store the hyperparameters and create empty slots for the weights W and the bias b
    def __init__(self, learning_rate, convergence_tol=1e-6):
        self.learning_rate = learning_rate
        self.convergence_tol = convergence_tol
        self.W = None
        self.b = None

    # Start from tiny random weights and a zero bias
    def initialize_parameters(self, n_features):
        self.W = np.random.randn(n_features) * 0.01
        self.b = 0

    # Forward pass (prediction): y_hat = X . W + b
    def forward(self, X):
        return np.dot(X, self.W) + self.b

    # Cost = mean squared error / 2 (the 1/2 makes the gradient simpler)
    def compute_cost(self, predictions):
        m = len(predictions)
        cost = np.sum(np.square(predictions - self.y)) / (2 * m)
        return cost

    # Backward pass: gradients of the cost with respect to W and b
    def backward(self, predictions):
        m = len(predictions)
        self.dW = np.dot(predictions - self.y, self.X) / m
        self.db = np.sum(predictions - self.y) / m

    # Train the model: repeat forward -> cost -> gradients -> update
    def fit(self, X, y, iterations, plot_cost=True):
        self.X = X
        self.y = y
        self.initialize_parameters(X.shape[1])
        costs = []

        # Each pass of this loop is one iteration of gradient descent
        for i in range(iterations):
            # 1) Predict with the current W and b
            predictions = self.forward(X)
            # 2) Measure how wrong the predictions are
            cost = self.compute_cost(predictions)
            # 3) Compute the gradients (which direction increases the error)
            self.backward(predictions)
            # 4) Take a small step in the opposite direction; step size = learning rate
            self.W -= self.learning_rate * self.dW
            self.b -= self.learning_rate * self.db
            costs.append(cost)

            # Print progress every 100 iterations
            if i % 100 == 0:
                print(f'Iteration: {i}, Cost: {cost}')

            # Stop early when the cost barely changes (the model has converged)
            if i > 0 and abs(costs[-1] - costs[-2]) < self.convergence_tol:
                print(f'Converged after {i} iterations.')
                break

        # Plot the cost curve to confirm that the model is learning
        if plot_cost:
            fig = px.line(y=costs, title="Cost vs Iteration", template="plotly_dark")
            fig.update_layout(xaxis=dict(title="Iterations"), yaxis=dict(title="Cost"))
            fig.show()

    # Predict for new data with the learned W and b
    def predict(self, X):
        return self.forward(X)

    # Save the learned parameters to a file with pickle
    def save_model(self, filename=None):
        model_data = {
            'learning_rate': self.learning_rate,
            'convergence_tol': self.convergence_tol,
            'W': self.W,
            'b': self.b
        }
        with open(filename, 'wb') as file:
            pickle.dump(model_data, file)

    # Rebuild a model from a saved file
    @classmethod
    def load_model(cls, filename):
        with open(filename, 'rb') as file:
            model_data = pickle.load(file)
        loaded_model = cls(model_data['learning_rate'], model_data['convergence_tol'])
        loaded_model.W = model_data['W']
        loaded_model.b = model_data['b']
        return loaded_model

## 9. Model Training

We train with a learning rate of 0.01 and allow at most 10,000 iterations.

**What the output shows:**
- The cost starts around **138,000** (the model is far off at the beginning) and drops quickly: about 27,000 after 100 iterations and about 9,970 after 1,000.
- Training **stops early after 1,079 iterations** because the cost stopped improving (`Converged after 1079 iterations`).
- The cost curve falls steeply and then flattens, which is what healthy training looks like.

The learned `w` and `b` are in **standardized units**, because we trained on the standardized `Length1`. We convert them to centimetres and grams further below.

In [62]:
# Create a model with learning rate 0.01
lr = LinearRegression(0.01)

# Train for at most 10,000 iterations (it stops earlier if the cost stops improving)
lr.fit(X_train, y_train, 10000)

Iteration: 0, Cost: 138030.14596227952
Iteration: 100, Cost: 27128.965566058887
Iteration: 200, Cost: 12270.461475209022
Iteration: 300, Cost: 10279.723928241268
Iteration: 400, Cost: 10013.005558970995
Iteration: 500, Cost: 9977.270718577514
Iteration: 600, Cost: 9972.482976280497
Iteration: 700, Cost: 9971.841516124237
Iteration: 800, Cost: 9971.755573501068
Iteration: 900, Cost: 9971.744058936358
Iteration: 1000, Cost: 9971.742516218725
Converged after 1079 iterations.


In [63]:
# Save the learned parameters to a file, then load them back into a new model object
lr.save_model('fish_model.pkl')
model = LinearRegression.load_model('fish_model.pkl')

# W and b are in STANDARDIZED units (Length1 was standardized before training)
print("w (standardized):", model.W)
print("b:", model.b)

w (standardized): [323.58610257]
b: 389.11708523458896


## 10. Model Evaluation

We evaluate on the **test set** (fish the model never saw). Three metrics:

| Metric | Meaning |
|---|---|
| **MSE** | Average squared error (in grams squared). Lower is better. |
| **RMSE** | Square root of MSE, so it is in **grams**: the typical size of a prediction error. |
| **R²** | Share of the variation in weight that the model explains. 1 = perfect, 0 = no better than always guessing the average. |

**Result (simple model, test set):**
- RMSE $\approx$ **151.5 g**: predictions are typically off by about 150 g.
- R² $\approx$ **0.829**: Length1 alone explains about 83% of the variation in weight.

That is a decent result for a single feature, but an error of 150 g is large when the average fish weighs about 400 g. The curve we saw earlier is part of the reason.

In [64]:
class RegressionMetrics:
    """Evaluation metrics for regression, written from scratch."""

    @staticmethod
    def mean_squared_error(y_true, y_pred):
        # Average of the squared errors (unit: grams squared)
        return np.mean((y_true - y_pred) ** 2)

    @staticmethod
    def root_mean_squared_error(y_true, y_pred):
        # Square root of MSE, so the error is back in grams
        return np.sqrt(RegressionMetrics.mean_squared_error(y_true, y_pred))

    @staticmethod
    def r_squared(y_true, y_pred):
        # Share of the variance in y that the model explains (1 = perfect, 0 = no better than the mean)
        ss_total = np.sum((y_true - np.mean(y_true)) ** 2)      # total variation around the mean
        ss_residual = np.sum((y_true - y_pred) ** 2)            # variation the model failed to explain
        return 1 - (ss_residual / ss_total)

In [65]:
# Predict the weight of the unseen test fish
y_pred = model.predict(X_test)

# Compare predictions with the real weights
mse_value = RegressionMetrics.mean_squared_error(y_test, y_pred)
rmse_value = RegressionMetrics.root_mean_squared_error(y_test, y_pred)
r_squared_value = RegressionMetrics.r_squared(y_test, y_pred)

print(f"MSE:  {mse_value}")
print(f"RMSE: {rmse_value}")
print(f"R²:   {r_squared_value}")

MSE:  22946.18466325492
RMSE: 151.479981064347
R²:   0.8290668633126949


### The Fitted Line in Original Units

Because we trained on standardized `Length1`, we convert the parameters back:

$$\text{slope} = \frac{W}{\sigma} \qquad \text{intercept} = b - \frac{W\,\mu}{\sigma}$$

**Result:** $\text{Weight} \approx 32.64 \times \text{Length1} - 462.22$

- **Slope:** each extra centimetre of length adds about 32.6 g.
- **Intercept:** the line predicts a **negative** weight for fish shorter than about 14 cm, which is impossible. This is a clear sign that a straight line is the wrong shape for this relationship.

In [66]:
# Convert w and b from standardized units back to the original units (cm -> grams)
# Derivation: y = W*(x - mean)/std + b  =>  slope = W/std, intercept = b - W*mean/std
w_original = model.W[0] / X_std
b_original = model.b - model.W[0] * X_mean / X_std
print(f"Weight = {w_original:.4f} * Length1 + {b_original:.4f}")

Weight = 32.6403 * Length1 + -462.2248


In [67]:
# Convert the standardized test inputs back to centimetres for plotting
X_test_original = X_test[:, 0] * X_std + X_mean

# Scatter plot of the real test fish
fig = px.scatter(x=X_test_original, y=y_test, template='seaborn',
                 labels={'x': 'Length1 (cm)', 'y': 'Weight (g)'},
                 title='Linear Regression fit on the test set')

# Draw the fitted line across the full range of lengths
line_x = np.array([X.min(), X.max()])
fig.add_scatter(x=line_x, y=w_original * line_x + b_original, mode='lines', name='Fitted line')
fig.show()

---

# Multiple Linear Regression

Now we use all five measurements:

$$\text{Weight} = w_1 \cdot \text{Length1} + w_2 \cdot \text{Length2} + w_3 \cdot \text{Length3} + w_4 \cdot \text{Height} + w_5 \cdot \text{Width} + b$$

Our `LinearRegression` class needs **no changes**. It uses vectorized math, so an `X` of shape `(126, 5)` instead of `(126, 1)` automatically gives multiple regression.

In [68]:
# The 5 numeric measurements used as features in multiple linear regression
multi_features = ['Length1', 'Length2', 'Length3', 'Height', 'Width']

# Keep the 5 features and the target
multi_data = dataset[multi_features + ['Weight']]
multi_data = multi_data.dropna()
multi_data = multi_data[multi_data['Weight'] > 0]   # same cleaning step: remove Weight = 0

print(f"Shape: {multi_data.shape}")   # expected (158, 6)
multi_data.head()

Shape: (158, 6)


,Length1,Length2,Length3,Height,Width,Weight
0,23.2,25.4,30.0,11.5200,4.0200,242.0
1,24.0,26.3,31.2,12.4800,4.3056,290.0
2,23.9,26.5,31.1,12.3778,4.6961,340.0
3,26.3,29.0,33.5,12.7300,4.4555,363.0
4,26.5,29.0,34.0,12.4440,5.1340,430.0


### Correlation Between Features

The heatmap shows how strongly each pair of columns is related (1 = move together perfectly).

**What we see:** `Length1`, `Length2` and `Length3` are correlated at **0.99 or higher** with each other. They are essentially the same measurement taken in three ways. This is called **multicollinearity**.

Consequences:
- Predictions are usually still good.
- The **individual weights** of these three features become unreliable, because the model cannot tell which one deserves the credit.
- Gradient descent needs many more iterations to settle.

`Height` and `Width` are less correlated with the lengths, so they add genuinely new information.

In [69]:
# Correlation matrix: values near 1 mean two columns carry almost the same information
px.imshow(multi_data.corr().round(3), text_auto=True, template='seaborn',
          title='Correlation matrix')

### Prepare the Multi-Feature Data

Same steps as before: take the 5 features, split 80/20 with the same `random_state` (so we test on exactly the same 32 fish as the simple model), and standardize each column separately.

Shapes are now `(126, 5)` for training and `(32, 5)` for testing.

In [70]:
# Features (5 columns) and target as NumPy arrays
X_multi = multi_data[multi_features].values
y_multi = multi_data['Weight'].values

# Same split settings as before -> exactly the same 32 test fish as the simple model
X_multi_train, X_multi_test, y_multi_train, y_multi_test = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42)

# Standardize every column separately (axis=0 inside the function)
X_multi_train, X_multi_test, X_multi_mean, X_multi_std = standardize_data(X_multi_train, X_multi_test)

print(f"X_multi_train: {X_multi_train.shape}, X_multi_test: {X_multi_test.shape}")   # (126, 5) and (32, 5)

X_multi_train: (126, 5), X_multi_test: (32, 5)


### Train the Multiple Regression Model

Here we use a larger learning rate (0.1), a much tighter stopping tolerance (`1e-10`) and up to 30,000 iterations.

**Why:** because three features are almost identical, the cost decreases very slowly after the first few hundred iterations. In this run the model used all 30,000 iterations without stopping early, so it is close to, but not fully at, its optimum. The cost drops from about 138,000 to about **7,573**, clearly lower than the simple model's 9,971.

In [71]:
# Larger learning rate (0.1) and tighter stopping tolerance (1e-10):
# the three length features are almost identical, so the cost decreases very slowly
multi_lr = LinearRegression(0.1, convergence_tol=1e-10)
multi_lr.fit(X_multi_train, y_multi_train, 30000)

Iteration: 0, Cost: 138035.36630254687
Iteration: 100, Cost: 7750.358974406642
Iteration: 200, Cost: 7741.826636938183
Iteration: 300, Cost: 7736.652543661502
Iteration: 400, Cost: 7731.895909924791
Iteration: 500, Cost: 7727.317586644088
Iteration: 600, Cost: 7722.893501228557
Iteration: 700, Cost: 7718.616889818024
Iteration: 800, Cost: 7714.4825361718695
Iteration: 900, Cost: 7710.485510190893
Iteration: 1000, Cost: 7706.621061746969
Iteration: 1100, Cost: 7702.884606747303
Iteration: 1200, Cost: 7699.271720737419
Iteration: 1300, Cost: 7695.778133288999
Iteration: 1400, Cost: 7692.399722628117
Iteration: 1500, Cost: 7689.13251045427
Iteration: 1600, Cost: 7685.972656940292
Iteration: 1700, Cost: 7682.916455906558
Iteration: 1800, Cost: 7679.960330163433
Iteration: 1900, Cost: 7677.100827016057
Iteration: 2000, Cost: 7674.33461392578
Iteration: 2100, Cost: 7671.658474322889
Iteration: 2200, Cost: 7669.069303565241
Iteration: 2300, Cost: 7666.564105037781
Iteration: 2400, Cost: 7664.

In [72]:
# Save, then reload the multiple-regression model
multi_lr.save_model('multi_fish_model.pkl')
multi_model = LinearRegression.load_model('multi_fish_model.pkl')

# One weight per feature, in the order of multi_features (standardized units)
print("w (standardized):", multi_model.W)
print("b:", multi_model.b)

w (standardized): [ 476.48331436  -34.79201391 -219.66372438   84.7147495    54.52922908]
b: 389.1246031746031


### Evaluate the Multiple Regression Model

| | Simple (Length1) | Multiple (5 features) |
|---|---|---|
| RMSE | 151.5 g | **113.0 g** |
| R² | 0.829 | **0.905** |

Adding the other measurements reduced the typical error by about 38 g and explains about 90% of the variation in weight.

**Predicted vs actual plot:** points close to the dashed diagonal are accurate predictions. Most fish are close to it. The largest fish tend to fall below the line (under-predicted), and some very small fish get negative predictions, both caused by the linear model failing to follow the curve.

**Reading the weights:** `Length1` has a large positive weight while `Length3` has a negative one. Do not interpret that as "longer Length3 means lighter fish". It is a side effect of multicollinearity: the model splits the same information among nearly identical features.

In [73]:
# Predict the weight of the 32 unseen test fish using all 5 features
y_multi_pred = multi_model.predict(X_multi_test)

multi_mse = RegressionMetrics.mean_squared_error(y_multi_test, y_multi_pred)
multi_rmse = RegressionMetrics.root_mean_squared_error(y_multi_test, y_multi_pred)
multi_r2 = RegressionMetrics.r_squared(y_multi_test, y_multi_pred)

print(f"MSE:  {multi_mse}")
print(f"RMSE: {multi_rmse}")
print(f"R²:   {multi_r2}")

MSE:  12769.90432586113
RMSE: 113.0040013710184
R²:   0.9048730831007529


In [74]:
# Predicted vs actual weight: points close to the dashed diagonal are good predictions
fig = px.scatter(x=y_multi_test, y=y_multi_pred, template='seaborn',
                 labels={'x': 'Actual Weight (g)', 'y': 'Predicted Weight (g)'},
                 title='Multiple regression: predicted vs actual (test set)')

# Reference line: where predicted = actual
fig.add_scatter(x=[y_multi.min(), y_multi.max()], y=[y_multi.min(), y_multi.max()],
                mode='lines', name='Perfect prediction', line=dict(dash='dash'))
fig.show()

---

# Scikit-learn Implementation

To check that our from-scratch code is correct, we fit the same models with scikit-learn and compare.

**Key difference:** scikit-learn's `LinearRegression` does **not** use gradient descent. It finds the best weights directly with linear algebra, so it gives the exact optimum. Our iterative method should get very close to it. (scikit-learn's gradient-descent version is `SGDRegressor`.)

It also needs no learning rate, no iteration count and no standardization for the simple model.

## Simple Linear Regression (scikit-learn)

We fit scikit-learn's `LinearRegression` on `Length1` and compare it with our gradient-descent model in a table. We use the same split settings, so both models are tested on the same 32 fish.

In [75]:
# scikit-learn's model and metrics (renamed so they do not clash with our own class)
from sklearn.linear_model import LinearRegression as SklearnLinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# scikit-learn needs X as a 2-D array, hence the double brackets [['Length1']]
# Note: NOT standardized here, so w and b come out directly in original units
X_sk = data[['Length1']].values
y_sk = data['Weight'].values

# Same split settings -> same test fish as our own model
X_sk_train, X_sk_test, y_sk_train, y_sk_test = train_test_split(X_sk, y_sk, test_size=0.2, random_state=42)

# fit() solves for the best w and b directly (no gradient descent, no learning rate)
sk_lr = SklearnLinearRegression()
sk_lr.fit(X_sk_train, y_sk_train)

# Evaluate on the test set
y_sk_pred = sk_lr.predict(X_sk_test)
sk_mse = mean_squared_error(y_sk_test, y_sk_pred)
sk_rmse = np.sqrt(sk_mse)          # scikit-learn has no direct RMSE here, so take the square root of MSE
sk_r2 = r2_score(y_sk_test, y_sk_pred)

# coef_ is w, intercept_ is b
print(f"Weight = {sk_lr.coef_[0]:.4f} * Length1 + {sk_lr.intercept_:.4f}")
print(f"RMSE: {sk_rmse}")
print(f"R²:   {sk_r2}")

# Side-by-side comparison: our gradient descent vs scikit-learn
comparison = pd.DataFrame({
    'From scratch (GD)': [w_original, b_original, mse_value, rmse_value, r_squared_value],
    'scikit-learn':      [sk_lr.coef_[0], sk_lr.intercept_, sk_mse, sk_rmse, sk_r2],
}, index=['w (Length1)', 'b', 'MSE', 'RMSE', 'R²'])
comparison

Weight = 32.6409 * Length1 + -462.2337
RMSE: 151.47861358537986
R²:   0.8290699494817384


,From scratch (GD),scikit-learn
w (Length1),32.640297,32.640928
b,-462.224759,-462.233688
MSE,22946.184663,22945.770374
RMSE,151.479981,151.478614
R²,0.829067,0.829070


**Comparison of the simple models:**

| | From scratch | scikit-learn |
|---|---|---|
| w | 32.6403 | 32.6409 |
| b | -462.22 | -462.23 |
| RMSE | 151.480 | 151.479 |
| R² | 0.82907 | 0.82907 |

The results agree to three decimals, which confirms our gradient descent implementation is correct.

## Multiple Linear Regression (scikit-learn)

Now the same comparison with all five (standardized) features. Both models use the same standardized inputs, so their weights can be compared directly.

In [76]:
# Same scikit-learn model, now with all 5 standardized features
sk_multi_lr = SklearnLinearRegression()
sk_multi_lr.fit(X_multi_train, y_multi_train)

# Evaluate on the same 32 test fish
y_sk_multi_pred = sk_multi_lr.predict(X_multi_test)
sk_multi_mse = mean_squared_error(y_multi_test, y_sk_multi_pred)
sk_multi_rmse = np.sqrt(sk_multi_mse)
sk_multi_r2 = r2_score(y_multi_test, y_sk_multi_pred)

# Compare weights and metrics (both models use the same standardized features, so w is comparable)
multi_comparison = pd.DataFrame({
    'From scratch (GD)': list(multi_model.W) + [multi_model.b, multi_mse, multi_rmse, multi_r2],
    'scikit-learn':      list(sk_multi_lr.coef_) + [sk_multi_lr.intercept_, sk_multi_mse, sk_multi_rmse, sk_multi_r2],
}, index=[f'w ({f})' for f in multi_features] + ['b', 'MSE', 'RMSE', 'R²'])
multi_comparison

,From scratch (GD),scikit-learn
w (Length1),476.483314,636.026944
w (Length2),-34.792014,-217.750560
w (Length3),-219.663724,-198.355825
w (Height),84.714750,85.347844
w (Width),54.529229,56.671012
b,389.124603,389.124603
MSE,12769.904326,12829.160458
RMSE,113.004001,113.265884
R²,0.904873,0.904432


**Comparison of the multiple models:**

The metrics agree closely (RMSE 113.00 vs 113.27, R² 0.905 vs 0.904), and so do `b`, `Height` and `Width`. The weights of the three length features **differ**:

| | Length1 | Length2 | Length3 | Sum |
|---|---|---|---|---|
| From scratch | 476.5 | -34.8 | -219.7 | **222.0** |
| scikit-learn | 636.0 | -217.8 | -198.4 | **219.9** |

The individual numbers differ because of multicollinearity (and because our gradient descent had not fully converged), but their **sum is almost identical**. The model only needs the combined "length effect", so the predictions are nearly the same however that effect is shared among the three features.

This is a good reminder: **good predictions do not guarantee interpretable coefficients** when features are highly correlated.

---

# Final Comparison & Conclusions

| | Simple (Length1) | Multiple (5 features) |
|---|---|---|
| RMSE | 151.5 g | 113.0 g |
| R² | 0.829 | 0.905 |

**What we learned**

1. **Check the data first.** `describe()` revealed an impossible weight of 0 g that would have distorted the model.
2. **Standardize using training statistics only** to avoid data leakage, and keep the mean/std to convert results back.
3. **Gradient descent works:** our from-scratch model matched scikit-learn almost exactly for the simple regression.
4. **More informative features help:** adding Length2, Length3, Height and Width cut the error by about 25%.
5. **Multicollinearity** makes individual weights unreliable and slows training, even when predictions are good.
6. **A straight line is not the best shape here.** Weight grows roughly with the cube of length, which explains the negative predictions for small fish and the under-prediction for large fish.

**Possible improvements**
- Take the logarithm of weight and the measurements, which turns the curve into a straight line.
- Use only one of the three length features, or add `Species` as extra columns.

In [77]:
# Final comparison of the two models on the same test fish
summary = pd.DataFrame({
    'Simple (Length1)':      [rmse_value, r_squared_value],
    'Multiple (5 features)': [multi_rmse, multi_r2],
}, index=['RMSE (g)', 'R²'])
display(summary)

# Standardized weights are on the same scale, so their sizes can be compared
pd.Series(multi_model.W, index=multi_features).sort_values(key=abs, ascending=False)

,Simple (Length1),Multiple (5 features)
RMSE (g),151.479981,113.004001
R²,0.829067,0.904873


,0
Length1,476.483314
Length3,-219.663724
Height,84.714750
Width,54.529229
Length2,-34.792014
